# 08 · Speakers + balancing

* **cluster**: per-episode speakers are merged into global speakers (TitaNet voice embeddings, cosine ≥ `speakers.merge_similarity`) → estimated number of distinct speakers; gender per speaker.
* **balance**: final selection (quality + transcript + alignment [+ ASR agreement]) and a cap per speaker (`max_hours_per_speaker`, `max_speaker_share`), keeping each speaker's best clips spread over their episodes. Unseen speakers form the test split.

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
nb.run("cluster")

In [ ]:
spk = pd.DataFrame(json.loads(lay.speakers.read_text())["speakers"])
print(len(spk), "global speakers |", spk.gender.value_counts().to_dict())
px.bar(spk.head(50), x="id", y="hours", color="gender", title="Hours per speaker before balancing (50 largest)").show()

In [ ]:
nb.run("balance")

In [ ]:
print(json.dumps(json.loads((lay.final / "balance_summary.json").read_text()), indent=1))
final = nb.final_frame(lay)
px.bar(final.groupby("speaker_id").duration.sum().div(3600).sort_values(ascending=False).head(50), title="Hours per speaker after balancing").show()